# Q1: Health Outcome Disparities Across Geographic Tiers

**Question:** Do health outcomes (mortality, recovery, DALYs) differ meaningfully across urbanization tiers and countries?

**Sources:** `v_outcome_by_geography`, `clean_health`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from notebooks._helpers import (
    get_connection, set_plot_style, save_fig,
    effect_size_label, format_ci,
    TIER_COLORS, COLOR_RURAL, COLOR_PERIURBAN, COLOR_URBAN,
)

set_plot_style()
con = get_connection()
print('Connected to DuckDB')

## 1. Load Data

In [ ]:
# Aggregated view
geo_df = con.execute('SELECT * FROM v_outcome_by_geography').fetchdf()
print(f'v_outcome_by_geography: {len(geo_df):,} rows')

# Tier-level summary
tier_summary = con.execute("""
    SELECT
        urbanization_tier,
        COUNT(*) AS n,
        AVG(mortality_rate) AS avg_mortality,
        STDDEV_SAMP(mortality_rate) AS sd_mortality,
        AVG(recovery_rate) AS avg_recovery,
        AVG(dalys) AS avg_dalys
    FROM clean_health
    WHERE data_quality_flag = 'ok'
    GROUP BY urbanization_tier
    ORDER BY urbanization_tier
""").fetchdf()
tier_summary

## 2. Cohen's d — Effect Sizes Between Tiers

In [ ]:
def cohens_d(group1: pd.Series, group2: pd.Series) -> float:
    """Compute Cohen's d between two groups."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    pooled_sd = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    if pooled_sd == 0:
        return 0.0
    return (group1.mean() - group2.mean()) / pooled_sd


# Pull raw mortality by tier for effect size calculation
raw_by_tier = con.execute("""
    SELECT urbanization_tier, mortality_rate
    FROM clean_health
    WHERE data_quality_flag = 'ok'
""").fetchdf()

tiers = sorted(raw_by_tier['urbanization_tier'].unique())
pairs = [(tiers[i], tiers[j]) for i in range(len(tiers)) for j in range(i+1, len(tiers))]

effect_sizes = []
for t1, t2 in pairs:
    g1 = raw_by_tier.loc[raw_by_tier['urbanization_tier'] == t1, 'mortality_rate']
    g2 = raw_by_tier.loc[raw_by_tier['urbanization_tier'] == t2, 'mortality_rate']
    d = cohens_d(g1, g2)
    effect_sizes.append({'pair': f'{t1} vs {t2}', 'd': d, 'label': effect_size_label(d)})

es_df = pd.DataFrame(effect_sizes)
es_df

## 3. One-Way ANOVA — Variance Explained (eta-squared)

In [ ]:
groups = [g['mortality_rate'].values for _, g in raw_by_tier.groupby('urbanization_tier')]
f_stat, p_value = stats.f_oneway(*groups)

# Eta-squared
grand_mean = raw_by_tier['mortality_rate'].mean()
ss_between = sum(
    len(g) * (g['mortality_rate'].mean() - grand_mean) ** 2
    for _, g in raw_by_tier.groupby('urbanization_tier')
)
ss_total = ((raw_by_tier['mortality_rate'] - grand_mean) ** 2).sum()
eta_sq = ss_between / ss_total if ss_total > 0 else 0

print(f'ANOVA F = {f_stat:.4f}, p = {p_value:.2e}')
print(f'Eta-squared = {eta_sq:.6f} (variance explained by tier)')
print(f'Interpretation: Urbanization tier explains {eta_sq*100:.4f}% of mortality variance')
print(f'Note: With N~1M, even trivial differences reach statistical significance.')
print(f'      Effect size (eta-sq < 0.01) confirms no meaningful disparity.')

## 4. Visualization 1 — Grouped Bar Chart: Mortality by Tier with 95% CI

In [ ]:
tier_order = ['Rural', 'Peri-urban', 'Urban']
tier_plot = tier_summary.set_index('urbanization_tier').reindex(tier_order)
ci_half = 1.96 * tier_plot['sd_mortality'] / np.sqrt(tier_plot['n'])

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    tier_order,
    tier_plot['avg_mortality'],
    yerr=ci_half,
    capsize=6,
    color=[TIER_COLORS[t] for t in tier_order],
    edgecolor='white',
    linewidth=1.5,
)

for bar, m, ci in zip(bars, tier_plot['avg_mortality'], ci_half):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + ci + 0.01,
            format_ci(m, ci), ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Average Mortality Rate (%)')
ax.set_title('Q1: Mortality Rate by Urbanization Tier (95% CI)')
ax.set_ylim(bottom=0)
save_fig(fig, 'q1_bar_mortality_by_tier')
plt.show()

## 5. Visualization 2 — Heatmap: Country x Tier Mortality

In [ ]:
# Pivot: country x tier
pivot_data = geo_df.groupby(['country', 'urbanization_tier'])['avg_mortality'].mean().reset_index()
pivot = pivot_data.pivot(index='country', columns='urbanization_tier', values='avg_mortality')
pivot = pivot.reindex(columns=['Rural', 'Peri-urban', 'Urban'])

fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(
    pivot, annot=True, fmt='.2f', cmap='YlOrRd',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg Mortality Rate (%)'}
)
ax.set_title('Q1: Average Mortality by Country and Urbanization Tier')
ax.set_ylabel('')
ax.set_xlabel('Urbanization Tier')
save_fig(fig, 'q1_heatmap_country_tier')
plt.show()

## 6. Visualization 3 — Violin Plot: Mortality Distribution by Tier

In [ ]:
# Sample for plotting (full data is ~1M rows)
sample = con.execute("""
    SELECT urbanization_tier, mortality_rate
    FROM clean_health
    WHERE data_quality_flag = 'ok'
    USING SAMPLE 50000
""").fetchdf()

fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(
    data=sample, x='urbanization_tier', y='mortality_rate',
    order=tier_order, palette=TIER_COLORS, inner='box', ax=ax,
)
ax.set_xlabel('Urbanization Tier')
ax.set_ylabel('Mortality Rate (%)')
ax.set_title('Q1: Mortality Distribution by Urbanization Tier')
save_fig(fig, 'q1_violin_mortality')
plt.show()

## 7. Visualization 4 — Effect Size Dot Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(es_df))
ax.scatter(es_df['d'], y_pos, s=120, color='#2c3e50', zorder=5)

# Reference lines for effect size thresholds
for threshold, label in [(0.2, 'small'), (0.5, 'medium'), (0.8, 'large')]:
    ax.axvline(threshold, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(-threshold, color='gray', linestyle='--', alpha=0.5)
    ax.text(threshold, len(es_df) - 0.3, label, ha='center', fontsize=8, color='gray')

ax.axvline(0, color='black', linewidth=0.8)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(es_df['pair'])
ax.set_xlabel("Cohen's d")
ax.set_title("Q1: Effect Sizes Between Urbanization Tiers (Mortality)")

for i, row in es_df.iterrows():
    ax.annotate(f'd={row["d"]:.4f} ({row["label"]})',
                (row['d'], i), textcoords='offset points',
                xytext=(10, 0), fontsize=9)

save_fig(fig, 'q1_effect_size_dotplot')
plt.show()

## 8. Key Finding

**Health outcome disparities across urbanization tiers are consistently ABSENT.**

- Cohen's d < 0.01 for all tier pairs — **negligible** effect sizes
- Eta-squared near zero — urbanization tier explains virtually none of the mortality variance
- The heatmap shows near-identical mortality (~5.05%) across all 20 countries and all 3 tiers
- With N ~ 1M, ANOVA may reach statistical significance, but the effect size confirms no practical disparity

This finding is itself informative: the data shows **no geographic health disparity**, which is atypical of real-world health data and suggests synthetic generation with uniform distributions.

In [ ]:
con.close()
print('Q1 analysis complete.')